# AdaIN Neural Style Transfer: Complete & Simplified Pipeline
### Arbitrary Artistic Style Transfer in Real-Time with PyTorch & GPU Acceleration
**Author**: Senior AI/ML Engineer & Student Architect  
**Objective**: Clear, modular, and interview-ready implementation of **AdaIN** (Huang & Belongie, ICCV 2017) and **Perceptual Neural Style Transfer** (Gatys et al.).

---
### Architecture Overview
```
[Content Image (c)] ----> [VGG19 Encoder] ----> f(c) ----\
                                                          +---> [AdaIN Bottleneck] ---> t ---> [Trained Decoder] ---> [Stylized Output]
[Style Image (s)]   ----> [VGG19 Encoder] ----> f(s) ----/
```


---
## SECTION 1: Imports and Configuration
- Imports PyTorch, Torchvision, PIL, and Matplotlib.
- Detects CUDA GPU availability (e.g., NVIDIA GeForce RTX 4060 Laptop GPU).
- Configures random seeds for reproducible experiments.


In [ ]:
# SECTION 1: Imports and Configuration
import os
import time
from pathlib import Path

# Fix Windows OpenMP duplicate runtime issue if present
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
import torch.nn as nn
from PIL import Image
import matplotlib.pyplot as plt
from torchvision.transforms import functional as TF

# Import refactored project library
import adain_model as am

# Detect compute device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

print("=" * 50)
print(f"PyTorch Version : {torch.__version__}")
print(f"Active Device   : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU Model       : {torch.cuda.get_device_name(0)}")
    print(f"VRAM Available  : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
print("=" * 50)


---
## SECTION 2: Project Paths and Dataset Validation
- Sets up project root, dataset directories, checkpoint paths, and output folder.
- Scans and counts available content images (COCO) and style artworks (WikiArt).


In [ ]:
# SECTION 2: Project Paths and Dataset Validation
PROJECT_ROOT = Path("project")
CONTENT_DIR = PROJECT_ROOT / "dataset" / "test2017" / "test2017" / "test2017"
STYLE_DIR = PROJECT_ROOT / "dataset" / "style" / "train_8"
CHECKPOINT_PATH = PROJECT_ROOT / "checkpoints" / "adain_decoder_100_batches.pth"
OUTPUT_DIR = Path("generated_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Scan for valid image files
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}
content_files = [p for p in CONTENT_DIR.glob("*") if p.suffix.lower() in IMAGE_EXTENSIONS]
style_files = [p for p in STYLE_DIR.glob("*") if p.suffix.lower() in IMAGE_EXTENSIONS]

print(f"Content Images Found : {len(content_files):,}")
print(f"Style Artworks Found : {len(style_files):,}")
print(f"Trained Checkpoint   : {CHECKPOINT_PATH.name} (Exists: {CHECKPOINT_PATH.exists()})")
print(f"Results Directory    : {OUTPUT_DIR.resolve()}")


---
## SECTION 3: Image Loading and Preprocessing
- Loads images from disk, converts to RGB mode, and fits to target resolution using bicubic interpolation.
- Converts to PyTorch float32 tensors normalized in the range $[0.0, 1.0]$.


In [ ]:
# SECTION 3: Image Loading and Preprocessing
# Select a sample content and style image pair
sample_content_path = content_files[0]  # e.g., 000000000001.jpg (Taxi) or 000000006499.jpg (Train)
sample_style_path = style_files[0]      # e.g., 8.jpg (Horses) or 80695.jpg (Baroque)

# Preprocess to 384x384 resolution on active device
c_tensor = am.load_and_preprocess_image(sample_content_path, size=384, device=DEVICE)
s_tensor = am.load_and_preprocess_image(sample_style_path, size=384, device=DEVICE)

print(f"Content Image : {sample_content_path.name} | Shape: {tuple(c_tensor.shape)} | Range: [{c_tensor.min():.2f}, {c_tensor.max():.2f}]")
print(f"Style Image   : {sample_style_path.name}   | Shape: {tuple(s_tensor.shape)} | Range: [{s_tensor.min():.2f}, {s_tensor.max():.2f}]")


---
## SECTION 4: VGG19 Encoder
- Loads pretrained VGG19 network with frozen weights.
- Standardizes input images using ImageNet statistics:
  $$\text{mean} = [0.485, 0.456, 0.406], \quad \text{std} = [0.229, 0.224, 0.225]$$
- Slices feature activations:
  - `relu1_1` (64 channels) & `relu2_1` (128 channels): low-level colors and edges
  - `relu3_1` (256 channels) & `relu4_1` (512 channels): complex textures and styles
  - `relu4_2` (512 channels): high-level content semantics


In [ ]:
# SECTION 4: VGG19 Encoder
encoder = am.VGG19Encoder().to(DEVICE).eval()

with torch.inference_mode():
    f1, f2, f3, f4 = encoder(c_tensor)

print("VGG19 Extracted Feature Map Dimensions:")
print(f"  relu1_1 : {tuple(f1.shape)} (fine textures & color)")
print(f"  relu2_1 : {tuple(f2.shape)} (brushstrokes & contours)")
print(f"  relu3_1 : {tuple(f3.shape)} (complex geometric patterns)")
print(f"  relu4_1 : {tuple(f4.shape)} (deep feature representation)")


---
## SECTION 5: Decoder and Checkpoint Loading
- Reconstructs 3-channel RGB image from 512-channel deep features (`relu4_1`).
- Uses 7 convolutional layers and nearest-neighbor 2x upsampling.
- Restores trained weights from `adain_decoder_100_batches.pth`.


In [ ]:
# SECTION 5: Decoder Architecture and Checkpoint Restoration
decoder = am.Decoder().to(DEVICE).eval()

# Load trained checkpoint weights
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True)
decoder.load_state_dict(checkpoint["decoder_state_dict"])

print("Trained Decoder successfully loaded!")
print(f"  Checkpoint File : {CHECKPOINT_PATH.name}")
print(f"  Average Loss    : {checkpoint.get('average_loss', 'N/A'):.4f}")
print(f"  Decoder State   : Evaluation Mode (eval=True)")


---
## SECTION 6: Adaptive Instance Normalization (AdaIN)
- Mathematically aligns the channel-wise mean and variance of content features to match style features:
  $$\text{AdaIN}(x, y) = \sigma(y) \left( \frac{x - \mu(x)}{\sigma(x)} \right) + \mu(y)$$
- Allows arbitrary artistic style transfer without retraining on individual styles.


In [ ]:
# SECTION 6: Adaptive Instance Normalization
with torch.inference_mode():
    sf4 = encoder(s_tensor)[-1]
    target_adain_feat = am.adaptive_instance_normalization(f4, sf4)

print(f"Content Feature Shape : {tuple(f4.shape)}")
print(f"Style Feature Shape   : {tuple(sf4.shape)}")
print(f"AdaIN Target Shape    : {tuple(target_adain_feat.shape)}")


---
## SECTION 7: Basic Neural Style Transfer (AdaIN Inference)
- Fast, feed-forward inference (~20ms on GPU).
- Supports style strength control ($\alpha \in [0.0, 1.0]$):
  $$t_{\text{blend}} = (1 - \alpha) f(c) + \alpha \text{AdaIN}(f(c), f(s))$$


In [ ]:
# SECTION 7: Basic Neural Style Transfer (AdaIN Feed-Forward)
t_start = time.time()
with torch.inference_mode():
    adain_output = am.style_transfer(encoder, decoder, c_tensor, s_tensor, alpha=1.0)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
elapsed_ms = (time.time() - t_start) * 1000

print(f"AdaIN Feed-Forward Completed in {elapsed_ms:.2f} ms on {DEVICE}!")
print(f"Output Tensor Shape: {tuple(adain_output.shape)}")

# Dynamic range contrast normalization
adain_norm = (adain_output - adain_output.min()) / (adain_output.max() - adain_output.min() + 1e-8)


---
## SECTION 8: Loss Functions
- **Content Loss**: $L_c = \| f(g(t)) - t \|_2$ (preserves semantic structure).
- **Style Loss**: Multi-layer statistical alignment across `relu1_1` to `relu4_1`:
  $$L_s = \sum_{i} w_i \left( \| \mu(g_i) - \mu(s_i) \|_2 + \| \sigma(g_i) - \sigma(s_i) \|_2 \right)$$
- **Gram Matrix & Total Variation**: Captures feature correlation patterns and removes high-frequency digital noise.


In [ ]:
# SECTION 8: Loss Functions
with torch.no_grad():
    g_feats = encoder(adain_norm)
    c_loss = am.calc_content_loss(g_feats[-1], target_adain_feat)
    s_loss = am.calc_style_loss(list(g_feats), list(encoder(s_tensor)))
    tv_loss = am.total_variation_loss(adain_norm)

print(f"Content Loss (L_c) : {c_loss.item():.4f}")
print(f"Style Loss (L_s)   : {s_loss.item():.4f}")
print(f"TV Smoothness Loss : {tv_loss.item():.2f}")


---
## SECTION 9: High-Fidelity Perceptual Style Transfer
- Solves under-styling of early checkpoints by optimizing pixel gradients directly using multi-scale Gram matrices.
- Transfers the full color palette, chiaroscuro shadows, and rich canvas brushstrokes of the style artwork in ~4-6 seconds on GPU.


In [ ]:
# SECTION 9: High-Fidelity Perceptual Style Transfer
print(f"Executing High-Fidelity Perceptual Style Transfer on {DEVICE}...")
t_start = time.time()
vivid_output = am.perceptual_style_transfer(
    content_tensor=c_tensor,
    style_tensor=s_tensor,
    steps=180,
    lr=0.035,
    style_weight=4e6,
    content_weight=0.6,
    tv_weight=1e-5,
    color_transfer=True,
    device=DEVICE,
    encoder=encoder
)
if DEVICE.type == "cuda":
    torch.cuda.synchronize()
gpu_time = time.time() - t_start

print(f"High-Fidelity Rendering Completed in {gpu_time:.2f} seconds!")


---
## SECTION 10: Image Generation and Saving
- Saves high-resolution stylized output images directly into `generated_results/`.


In [ ]:
# SECTION 10: Image Generation and Saving
out_file_adain = OUTPUT_DIR / f"final_adain_{sample_content_path.stem}_{sample_style_path.stem}.png"
out_file_vivid = OUTPUT_DIR / f"final_vivid_{sample_content_path.stem}_{sample_style_path.stem}.png"

TF.to_pil_image(adain_norm[0].clamp(0, 1).cpu()).save(out_file_adain)
TF.to_pil_image(vivid_output[0].clamp(0, 1).cpu()).save(out_file_vivid)

print(f"Saved AdaIN Result : {out_file_adain}")
print(f"Saved Vivid Result : {out_file_vivid}")


---
## SECTION 11: Image Comparison and Evaluation
- Displays side-by-side comparison:
  1. Original Content Image
  2. Style Artwork
  3. High-Fidelity Stylized Output


In [ ]:
# SECTION 11: Image Comparison and Evaluation
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Content Image
axes[0].imshow(c_tensor[0].cpu().permute(1, 2, 0).clamp(0, 1))
axes[0].set_title(f"Content Image\n({sample_content_path.name})", fontsize=11, fontweight="bold")
axes[0].axis("off")

# 2. Style Artwork
axes[1].imshow(s_tensor[0].cpu().permute(1, 2, 0).clamp(0, 1))
axes[1].set_title(f"Style Artwork\n({sample_style_path.name})", fontsize=11, fontweight="bold")
axes[1].axis("off")

# 3. High-Fidelity Stylized Output
axes[2].imshow(vivid_output[0].cpu().permute(1, 2, 0).clamp(0, 1))
axes[2].set_title("Generated Stylized Image\n(True Colors & Canvas Textures)", fontsize=11, fontweight="bold")
axes[2].axis("off")

plt.tight_layout()
plt.show()

print("=" * 60)
print("PROJECT EXECUTION & VERIFICATION COMPLETE")
print(f"Device Used     : {DEVICE}")
print(f"Image Size      : 384x384")
print(f"AdaIN Latency   : {elapsed_ms:.2f} ms")
print(f"Vivid Latency   : {gpu_time:.2f} s")
print("=" * 60)
